# Siniestralidad vial — Ciudad de Chicago
## Fase 4: Modelado de riesgo relativo

**Cliente:** Oficina de Seguridad Vial, Departamento de Transporte de la Ciudad de Chicago (CDOT) — Observatorio Vial
**Proveedor:** Grupo 1 de Ciencia de Datos

El brief pide, *"si los datos lo permiten, un modelo simple que estime el riesgo relativo por zona y horario — no predicción en tiempo real, sino un criterio objetivo para priorizar dónde intervenir primero"*. La propuesta divide esta fase en dos objetivos:

1. **Evaluar la viabilidad** de un modelo de riesgo relativo por zona y horario.
2. **Construirlo y validarlo**, usando al Observatorio Vial como contraparte para asegurar que el resultado tenga sentido en el terreno.

### Las dos preguntas que responde el modelo

| Pregunta | Modelo | Para qué sirve |
|---|---|---|
| **¿Dónde y cuándo ocurren más siniestros graves?** | **Modelo A** — regresión de Poisson sobre la cantidad de graves por zona × horario | Es el **criterio de priorización**: dice en qué zona y franja se concentra el daño. Alimenta la asignación de patrullaje y obra |
| **Cuando ocurre un siniestro, ¿qué lo vuelve grave?** | **Modelo B** — regresión logística sobre *grave* a nivel de cada siniestro | Separa efectos que en la Fase 3 aparecían mezclados (por ejemplo: ¿la madrugada es más riesgosa o solo hay más peatones atropellados?). Permite ver qué zonas tienen siniestros **más severos de lo esperable** por su composición |

Además, se corrige el ranking de puntos críticos de la Fase 3 con el método **Empírico Bayesiano**, el estándar del *Highway Safety Manual* para no confundir un cruce con mala suerte con uno realmente peligroso.

Se mantienen las definiciones de la Fase 3: **grave (KSI)** = al menos una lesión fatal o incapacitante; período **2018–2024**.

## 0. Librerías y configuración

Solo se usan librerías ya presentes en el proyecto (`numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`, `seaborn`, `folium`). Las regresiones se ajustan con una función propia de mínimos cuadrados reponderados (IRLS), que es el algoritmo estándar de los modelos lineales generalizados y permite obtener errores estándar e intervalos de confianza sin dependencias adicionales.

In [ ]:
import json
import textwrap
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon
from matplotlib.path import Path as MplPath
from scipy import stats
from scipy.special import expit
from sklearn.metrics import brier_score_loss, roc_auc_score

from config import (
    ANIO_CORTE_VALIDACION,
    ANIO_DESDE,
    ANIO_HASTA,
    ANIO_PRUEBA_DESDE,
    N_ANIOS,
)
from preparacion import FRANJAS, cargar_base

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

# Misma paleta que la Fase 3
AZUL, AZUL_CLARO, ROJO, NARANJA, VERDE = "#2a78d6", "#86b6ef", "#e34948", "#eb6834", "#1baf7a"
GRIS, TINTA, TINTA2, GRILLA, FONDO = "#898781", "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"
CMAP_AZUL = sns.blend_palette(["#f4f8fd", "#86b6ef", "#2a78d6", "#0d366b"], as_cmap=True)
CMAP_ROJO = sns.blend_palette(["#fdf3f2", "#f2a3a2", "#e34948", "#8f1d1c"], as_cmap=True)
CMAP_DIV = sns.blend_palette(["#2a78d6", "#86b6ef", "#f4f3ee", "#f2a3a2", "#e34948"], as_cmap=True)

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "figure.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.edgecolor": "#c3c2b7", "axes.labelcolor": TINTA2,
    "xtick.color": TINTA2, "ytick.color": TINTA2, "text.color": TINTA,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRILLA, "grid.linewidth": 0.6, "axes.axisbelow": True,
})

### Funciones auxiliares

- `matriz_diseno`: arma la matriz de variables (*dummies*) a partir de variables categóricas, indicando la **categoría de referencia** de cada una.
- `ajustar_glm`: ajusta una regresión de **Poisson** (conteos, con *offset* de exposición) o **logística** (sí/no) por IRLS y devuelve coeficientes, errores estándar y la matriz de covarianza. Admite un factor de **sobredispersión** para corregir los errores estándar (cuasi-Poisson).
- `tabla_efectos`: convierte los coeficientes en **razones** (riesgo relativo u *odds ratio*) con su IC 95 %.
- `ic_poisson`: intervalo exacto para un conteo (o una razón observado/esperado).
- `heatmap`: igual que en la Fase 3.

In [ ]:
def matriz_diseno(data, categoricas, numericas=()):
    # categoricas: {columna: categoría de referencia}. Devuelve DataFrame con intercepto + dummies.
    partes = [pd.Series(1.0, index=data.index, name="intercepto")]
    for col, ref in categoricas.items():
        d = pd.get_dummies(data[col].astype(str), prefix=col, prefix_sep=" = ", dtype=float)
        partes.append(d.drop(columns=f"{col} = {ref}"))
    for col in numericas:
        partes.append(data[col].astype(float))
    return pd.concat(partes, axis=1)


def ajustar_glm(X, y, familia="poisson", offset=None, max_iter=100, tol=1e-9):
    nombres = list(X.columns)
    X, y = X.to_numpy(float), np.asarray(y, float)
    off = np.zeros(len(y)) if offset is None else np.asarray(offset, float)
    beta = np.zeros(X.shape[1])
    beta[0] = np.log(y.sum() / np.exp(off).sum()) if familia == "poisson" else np.log(y.mean() / (1 - y.mean()))
    for _ in range(max_iter):
        eta = X @ beta + off
        if familia == "poisson":
            mu = np.exp(eta); w = mu
        else:
            mu = expit(eta); w = np.clip(mu * (1 - mu), 1e-10, None)
        z = eta - off + (y - mu) / w
        xtw = X.T * w
        nuevo = np.linalg.solve(xtw @ X, xtw @ z)
        if np.max(np.abs(nuevo - beta)) < tol:
            beta = nuevo
            break
        beta = nuevo
    eta = X @ beta + off
    mu = np.exp(eta) if familia == "poisson" else expit(eta)
    w = mu if familia == "poisson" else mu * (1 - mu)
    cov = np.linalg.inv((X.T * w) @ X)
    gl = len(y) - X.shape[1]
    if familia == "poisson":
        dev = 2 * np.sum(np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0) - (y - mu))
        pearson = np.sum((y - mu) ** 2 / mu)
    else:
        dev = -2 * np.sum(y * np.log(mu) + (1 - y) * np.log(1 - mu))
        pearson = np.sum((y - mu) ** 2 / (mu * (1 - mu)))
    return {"beta": pd.Series(beta, nombres), "cov": pd.DataFrame(cov, nombres, nombres), "mu": mu,
            "deviance": dev, "pearson": pearson, "gl": gl, "familia": familia, "columnas": nombres}


def predecir(modelo, X, offset=None):
    eta = X[modelo["columnas"]].to_numpy(float) @ modelo["beta"].to_numpy() + (0 if offset is None else np.asarray(offset))
    return np.exp(eta) if modelo["familia"] == "poisson" else expit(eta)


def tabla_efectos(modelo, escala=1.0):
    # escala: factor de sobredispersión (phi) que infla los errores estándar
    b = modelo["beta"]
    se = np.sqrt(np.diag(modelo["cov"]) * escala)
    t = pd.DataFrame({"coef": b, "se": se})
    t["razon"] = np.exp(t["coef"])
    t["ic_inf"], t["ic_sup"] = np.exp(t["coef"] - 1.96 * t["se"]), np.exp(t["coef"] + 1.96 * t["se"])
    t["p_valor"] = 2 * stats.norm.sf(np.abs(t["coef"] / t["se"]))
    return t.drop(index="intercepto")


def ic_poisson(obs, esp=1.0):
    obs = np.asarray(obs, float)
    lo = np.where(obs > 0, stats.chi2.ppf(0.025, 2 * obs) / 2, 0)
    hi = stats.chi2.ppf(0.975, 2 * obs + 2) / 2
    return lo / esp, hi / esp


def heatmap(tabla, titulo, cmap, fmt=".0f", ax=None, cbar_label="", mask=None, **kw):
    ax = ax or plt.gca()
    sns.heatmap(tabla, cmap=cmap, annot=True, fmt=fmt, linewidths=1, linecolor=FONDO, mask=mask,
                cbar_kws={"label": cbar_label, "shrink": 0.8}, ax=ax, annot_kws={"fontsize": 7}, **kw)
    ax.set_title(titulo)
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_xticklabels([textwrap.fill(t.get_text(), 16) for t in ax.get_xticklabels()], rotation=0, fontsize=8)
    ax.set_yticklabels([t.get_text() for t in ax.get_yticklabels()], rotation=0)
    return ax

## 1. Carga de datos y asignación de zonas

La preparación es la misma que en la Fase 3 (nombres de columnas, indicadores de severidad, franjas, agrupamientos de clima e iluminación, familias de causa y palancas). Para no duplicar código, quedó en el módulo [`preparacion.py`](preparacion.py), que el tablero de la Fase 5 también podrá reutilizar.

### ¿Qué es una "zona"?

Se usan las **77 áreas comunitarias** (*Community Areas*) oficiales de Chicago, descargadas del portal de datos abiertos de la Ciudad. Se eligieron porque:

- son la división territorial que el Concejo Municipal, la prensa y los vecinos ya conocen (*Austin*, *Loop*, *Near West Side*…), así que los resultados se comunican sin traducción;
- sus límites son estables en el tiempo, lo que permite medir la línea base y el impacto a 12 meses sobre la misma unidad;
- tienen suficiente volumen de siniestros graves para que las tasas sean estables (se verifica en la sección 2).

Cada siniestro se asigna al área comunitaria que contiene sus coordenadas (prueba de punto en polígono).

In [ ]:
df = cargar_base()
print(f"Siniestros {ANIO_DESDE}-{ANIO_HASTA}: {len(df):,} · graves: {df['grave'].sum():,}")

RUTA_AREAS = Path("areas_comunitarias.geojson")
if not RUTA_AREAS.exists():
    url = "https://data.cityofchicago.org/api/geospatial/igwz-8jzy?method=export&format=GeoJSON"
    RUTA_AREAS.write_bytes(requests.get(url, timeout=120).content)
areas_geo = json.loads(RUTA_AREAS.read_text(encoding="utf-8"))

areas = pd.DataFrame([{
    "area": int(f["properties"]["area_numbe"]),
    "zona": f["properties"]["community"].title().replace("Ohare", "O'Hare"),
    "km2": float(f["properties"]["shape_area"]) * 0.09290304 / 1e6,  # pies² -> km²
} for f in areas_geo["features"]]).set_index("area").sort_index()

# Asignación de cada siniestro a su área comunitaria (punto en polígono)
con_coord = df["latitud"].notna()
pts = df.loc[con_coord, ["longitud", "latitud"]].to_numpy()
asignada = np.zeros(len(pts), dtype=int)
for f in areas_geo["features"]:
    n_area = int(f["properties"]["area_numbe"])
    for poligono in f["geometry"]["coordinates"]:
        dentro = MplPath(np.asarray(poligono[0])).contains_points(pts)
        asignada[dentro & (asignada == 0)] = n_area
df["area"] = 0
df.loc[con_coord, "area"] = asignada
df["zona"] = df["area"].map(areas["zona"])

sin_zona = df["zona"].isna()
print(f"Siniestros asignados a un área comunitaria: {(~sin_zona).mean():.1%}")
print(f"Sin zona (sin coordenadas o fuera de los límites): {sin_zona.sum():,} "
      f"— % graves {df.loc[sin_zona, 'grave'].mean():.2%} vs. {df.loc[~sin_zona, 'grave'].mean():.2%} del resto")
dz = df[~sin_zona].copy()

### Definición de "horario"

Se combinan las **4 franjas** de la Fase 3 con el **tipo de día** (hábil o fin de semana), lo que da **8 horarios**. Es una grilla que el área de patrullaje puede usar directamente (turnos de 6 h) y que captura los dos patrones temporales más fuertes del diagnóstico: madrugada/noche y fin de semana.

Como los fines de semana tienen menos días que la semana hábil, comparar *cantidades* sería engañoso. El modelo trabaja con **tasas por hora de exposición temporal**: cuántos siniestros graves ocurren por cada hora que la ciudad pasa en ese horario.

In [ ]:
dz["tipo_dia"] = np.where(dz["fin_de_semana"], "Fin de semana", "Hábil")
TIPOS_DIA = ["Hábil", "Fin de semana"]
FRANJA_CORTA = dict(zip(FRANJAS, ["Madrugada 0-6", "Mañana 6-12", "Tarde 12-18", "Noche 18-24"]))
HORARIOS = [f"{t} · {FRANJA_CORTA[f]}" for t in TIPOS_DIA for f in FRANJAS]
dz["horario"] = pd.Categorical(dz["tipo_dia"] + " · " + dz["franja"].map(FRANJA_CORTA).astype(str),
                               categories=HORARIOS, ordered=True)

# Horas de calendario de cada horario en el período (cada franja dura 6 h)
dias = pd.date_range(f"{ANIO_DESDE}-01-01", f"{ANIO_HASTA}-12-31", freq="D")
dias_por_tipo = {"Hábil": int((dias.dayofweek < 5).sum()), "Fin de semana": int((dias.dayofweek >= 5).sum())}
horas = pd.Series({h: dias_por_tipo[h.split(" · ")[0]] * 6 for h in HORARIOS}, name="horas")
pd.DataFrame({"días en el período": [dias_por_tipo[h.split(" · ")[0]] for h in HORARIOS], "horas": horas})

## 2. Evaluación de viabilidad

Antes de ajustar un modelo hay que responder si los datos alcanzan para que el resultado sea **estable** (que no cambie por azar de un año a otro) y **útil** (que tenga la resolución que necesita el cliente). Se revisan cuatro condiciones.

### 2.1 Resolución espacial: ¿cuántos graves hay por unidad?

Los siniestros graves son el 3,4 % del total. Si la zona es muy chica, la mayoría tendrá 0, 1 o 2 graves, y una tasa calculada sobre eso es ruido. Se compara la grilla de ~150 m de la Fase 3, una grilla de ~1 km y las áreas comunitarias.

In [ ]:
def resumen_resolucion(claves, nombre):
    g = dz.groupby(claves).agg(siniestros=("grave", "size"), graves=("grave", "sum"))
    por_horario = dz.groupby([claves, "horario"], observed=False)["grave"].sum() if isinstance(claves, str) else None
    return {
        "unidad": nombre, "unidades con siniestros": len(g),
        "graves por unidad (mediana, 7 años)": g["graves"].median(),
        "% unidades con 0 graves": (g["graves"] == 0).mean() * 100,
        "% unidades con < 10 graves": (g["graves"] < 10).mean() * 100,
        "graves por unidad × horario (mediana)": por_horario.median() if por_horario is not None else np.nan,
    }

dz["celda_150m"] = ((dz["latitud"] / 0.0015).round().astype(int).astype(str) + "_" +
                    (dz["longitud"] / 0.0015).round().astype(int).astype(str))
dz["celda_1km"] = ((dz["latitud"] / 0.009).round().astype(int).astype(str) + "_" +
                   (dz["longitud"] / 0.012).round().astype(int).astype(str))
viabilidad = pd.DataFrame([
    resumen_resolucion("celda_150m", "Celda ~150 m (Fase 3)"),
    resumen_resolucion("celda_1km", "Celda ~1 km"),
    resumen_resolucion("zona", "Área comunitaria (77)"),
]).set_index("unidad")
viabilidad.round(1)

### 2.2 Eventos por parámetro y supuestos del modelo

In [ ]:
n_param_A = 1 + (dz["zona"].nunique() - 1) + (len(HORARIOS) - 1)
n_param_B_aprox = 30  # se confirma en la sección 4
chequeo = pd.DataFrame([
    ("Graves con zona asignada", f"{dz['grave'].sum():,}", "—"),
    ("Modelo A: celdas zona × horario", f"{dz['zona'].nunique() * len(HORARIOS):,}", "—"),
    ("Modelo A: parámetros", n_param_A, "—"),
    ("Modelo A: graves por parámetro", f"{dz['grave'].sum() / n_param_A:,.0f}", "≥ 10–20 (regla práctica)"),
    ("Modelo B: siniestros (filas)", f"{len(dz):,}", "—"),
    ("Modelo B: graves por parámetro (≈)", f"{dz['grave'].sum() / n_param_B_aprox:,.0f}", "≥ 10–20 (regla práctica)"),
    ("Años completos para validar fuera de muestra", f"{ANIO_DESDE}–{ANIO_CORTE_VALIDACION} vs. {ANIO_PRUEBA_DESDE}–{ANIO_HASTA}", "al menos 1 año de prueba"),
], columns=["Condición", "Valor", "Umbral de referencia"])
chequeo

### 2.3 Dictamen de viabilidad

| Condición | Resultado | Conclusión |
|---|---|---|
| **Ubicación** | 99,4 % de los siniestros georreferenciados (Fase 3) y >99 % asignados a un área comunitaria | ✅ Viable |
| **Resolución espacial** | En la grilla de 150 m la mayoría de las celdas tiene 0 graves: sirve para *ubicar* puntos, no para *estimar tasas*. Las áreas comunitarias tienen decenas de graves cada una | ✅ Viable a nivel área comunitaria. A nivel de cruce se usa un método de corrección específico (sección 6) |
| **Eventos por parámetro** | Muy por encima del mínimo en ambos modelos | ✅ Estimaciones estables |
| **Validación** | 7 años completos permiten ajustar con 2018–2022 y probar con 2023–2024 | ✅ Se puede validar fuera de muestra |
| **Exposición (flujo vehicular)** | No disponible | ⚠️ El riesgo es **por hora de calendario y por siniestro**, no por vehículo-km. Una zona con mucho tránsito tendrá más graves aunque cada viaje no sea más peligroso. Para *priorizar dónde se concentra el daño* esto es lo que importa; para comparar la *peligrosidad intrínseca* de dos calles, no alcanza |
| **Causa** | 30 % sin determinar y asignada por el oficial | ⚠️ El modelo B incluye "sin determinar" como categoría propia, para no perder esos casos ni sesgar la muestra |

**Dictamen: el modelo es viable**, con la zona definida como área comunitaria y el horario como franja × tipo de día. Los resultados deben leerse como *dónde y cuándo se concentra el daño*, no como *qué tan peligroso es circular por ahí*.

## 3. Modelo A — Frecuencia de siniestros graves por zona y horario

### 3.1 Planteo

Para cada combinación **zona × horario** (77 × 8 = 616 celdas) se cuenta la cantidad de siniestros graves en el período y se ajusta una **regresión de Poisson**:

$$\log(\text{graves esperados}_{z,h}) = \beta_0 + \text{efecto zona}_z + \text{efecto horario}_h + \log(\text{horas}_h)$$

- El término $\log(\text{horas}_h)$ (*offset*) hace que el modelo compare **tasas por hora** y no cantidades.
- El modelo es **multiplicativo**: el riesgo de una celda es *riesgo de la zona × riesgo del horario*. Eso es justamente un **riesgo relativo** fácil de explicar: *"Austin tiene 3 veces el promedio de la ciudad, y la madrugada de fin de semana 1,5 veces el promedio horario; la combinación, ≈ 4,5 veces"*.
- Al tener solo efectos principales, cada celda "toma prestada" información de toda su zona y de todo su horario, lo que la hace mucho más estable que contar graves celda por celda.

In [ ]:
panel = (dz.groupby(["zona", "horario"], observed=False)
           .agg(graves=("grave", "sum"), siniestros=("grave", "size"), con_heridos=("con_heridos", "sum"))
           .reset_index())
panel["horas"] = panel["horario"].map(horas).astype(float)
panel["horario"] = panel["horario"].astype(str)

# Referencias: la zona y el horario de mayor volumen (no cambian los resultados, solo la parametrización)
REF_ZONA = dz["zona"].value_counts().index[0]
REF_HORARIO = HORARIOS[2]  # Hábil · Tarde 12-18
XA = matriz_diseno(panel, {"zona": REF_ZONA, "horario": REF_HORARIO})
mod_A = ajustar_glm(XA, panel["graves"], "poisson", offset=np.log(panel["horas"]))
panel["esperado_A"] = mod_A["mu"]

phi_A = mod_A["pearson"] / mod_A["gl"]
print(f"Celdas: {len(panel)} · parámetros: {XA.shape[1]} · deviance: {mod_A['deviance']:,.1f} con {mod_A['gl']} g.l.")
print(f"Sobredispersión (chi² de Pearson / g.l.): {phi_A:.2f}")

**Sobredispersión.** Si el modelo describiera perfectamente los datos, el cociente *chi² de Pearson / grados de libertad* sería ≈ 1. Un valor mayor indica que hay más variación entre celdas que la que explica *zona × horario* por separado (por ejemplo, una zona con un problema específico de madrugada). Se tiene en cuenta de dos formas:

1. Los intervalos de confianza se **inflan** por $\sqrt{\phi}$ (enfoque *cuasi-Poisson*).
2. En la sección 3.4 se combina la predicción del modelo con lo observado en cada celda (Empírico Bayesiano), para no perder esos patrones específicos cuando son reales.

### 3.2 Riesgo relativo por horario

El riesgo relativo (RR) se expresa respecto de la **tasa promedio de la ciudad**: RR = 1 es el promedio, RR = 2 es el doble de siniestros graves por hora.

> Una propiedad útil de la regresión de Poisson con efectos principales: los totales que predice para cada zona y para cada horario **coinciden exactamente** con los observados. Por eso los RR de zona y de horario se pueden calcular directamente con los datos (y explicar sin mencionar el modelo). El aporte del modelo está en las **celdas zona × horario** (sección 3.4) y en su capacidad de anticipar el futuro (sección 3.5).

In [ ]:
# RR de cada horario respecto del promedio: tasa del horario (sumando zonas) / tasa promedio
rr_h = panel.groupby("horario").agg(graves=("graves", "sum"), horas=("horas", "first")).reindex(HORARIOS)
rr_h["graves_por_100h"] = rr_h["graves"] / rr_h["horas"] * 100
rr_h["RR"] = rr_h["graves"] / rr_h["horas"] / (rr_h["graves"].sum() / rr_h["horas"].sum())
# IC: intervalo exacto de Poisson del conteo, ensanchado por sqrt(phi) en escala log (cuasi-Poisson)
f_phi = np.sqrt(max(phi_A, 1))
lo, hi = ic_poisson(rr_h["graves"])
rr_h["IC_inf"] = rr_h["RR"] * (lo / rr_h["graves"]) ** f_phi
rr_h["IC_sup"] = rr_h["RR"] * (hi / rr_h["graves"]) ** f_phi

fig, ax = plt.subplots(figsize=(10, 3.8))
y = np.arange(len(HORARIOS))[::-1]
col = [ROJO if lo > 1 else (AZUL if hi < 1 else GRIS) for lo, hi in zip(rr_h["IC_inf"], rr_h["IC_sup"])]
ax.hlines(y, rr_h["IC_inf"], rr_h["IC_sup"], color=col, lw=2)
ax.scatter(rr_h["RR"], y, color=col, s=55, zorder=3)
for yi, (v, g) in zip(y, zip(rr_h["RR"], rr_h["graves_por_100h"])):
    ax.text(v, yi + 0.22, f"RR {v:.2f} · {g:.2f} graves cada 100 h", ha="center", fontsize=8, color=TINTA2)
ax.axvline(1, color=TINTA2, ls="--", lw=1)
ax.set_yticks(y, HORARIOS)
ax.set_xlabel("Riesgo relativo (1 = promedio de la ciudad)")
ax.set_title("Riesgo relativo de siniestro grave por horario (tasa por hora, IC 95%)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
rr_h.round(2)

### 3.3 Riesgo relativo por zona

In [ ]:
zonas = panel.groupby("zona").agg(graves=("graves", "sum"), siniestros=("siniestros", "sum"),
                                  con_heridos=("con_heridos", "sum"))
zonas = zonas.join(areas.set_index("zona")["km2"])
zonas["graves_por_anio"] = zonas["graves"] / N_ANIOS
zonas["RR_zona"] = zonas["graves"] / zonas["graves"].mean()
lo, hi = ic_poisson(zonas["graves"])
# IC cuasi-Poisson: se ensancha el intervalo exacto por sqrt(phi) en escala log
zonas["RR_IC_inf"] = zonas["RR_zona"] * (lo / zonas["graves"]) ** f_phi
zonas["RR_IC_sup"] = zonas["RR_zona"] * (hi / zonas["graves"]) ** f_phi
zonas["graves_km2_anio"] = zonas["graves_por_anio"] / zonas["km2"]
zonas["tasa_graves_%"] = zonas["graves"] / zonas["siniestros"] * 100
zonas = zonas.sort_values("graves", ascending=False)

TOP_Z = 20
t = zonas.head(TOP_Z).iloc[::-1]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 0.36 * TOP_Z + 1.5), sharey=True, gridspec_kw={"width_ratios": [1.2, 1]})
yy = np.arange(len(t))
col = [ROJO if l > 1 else (AZUL if h < 1 else GRIS) for l, h in zip(t["RR_IC_inf"], t["RR_IC_sup"])]
a1.hlines(yy, t["RR_IC_inf"], t["RR_IC_sup"], color=col, lw=2)
a1.scatter(t["RR_zona"], yy, color=col, s=45, zorder=3)
for yi, (v, g) in enumerate(zip(t["RR_zona"], t["graves_por_anio"])):
    a1.text(v, yi + 0.25, f"{v:.1f}× · {g:.0f}/año", ha="center", fontsize=7, color=TINTA2)
a1.axvline(1, color=TINTA2, ls="--", lw=1)
a1.set_yticks(yy, t.index)
a1.set_title("RR de siniestros graves (1 = zona promedio)")
a1.grid(axis="y", visible=False)
a2.barh(yy, t["graves_km2_anio"], color=AZUL, height=0.65)
for yi, v in enumerate(t["graves_km2_anio"]):
    a2.text(v, yi, f" {v:.2f}", va="center", fontsize=7, color=TINTA2)
a2.set_title("Densidad: graves por km² por año")
a2.grid(axis="y", visible=False)
fig.suptitle(f"Las {TOP_Z} zonas con más siniestros graves ({ANIO_DESDE}-{ANIO_HASTA})", x=0.01, ha="left", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

print(f"Las {TOP_Z} zonas con más graves (26% de las áreas) concentran el "
      f"{zonas.head(TOP_Z)['graves'].sum() / zonas['graves'].sum():.0%} de los graves.")

### 3.4 Riesgo por zona × horario

El modelo multiplicativo supone que todas las zonas tienen el **mismo perfil horario**. Se prueba si eso es cierto comparando con el modelo saturado (cada celda con su propia tasa), y se calcula para cada celda una estimación **Empírico Bayesiana (EB)**: un promedio ponderado entre lo que predice el modelo y lo que efectivamente se observó en la celda.

- Si la celda tiene pocos graves, la estimación EB queda cerca del modelo (no se le cree a un conteo chico).
- Si tiene muchos y se aparta sistemáticamente, la estimación EB se acerca a lo observado.

El peso se calcula a partir de la sobredispersión: $w = 1 / (1 + \hat\alpha \cdot \mu)$, con $\hat\alpha$ estimado por momentos.

In [ ]:
# Test de interacción zona × horario (cuasi-verosimilitud): ¿las zonas difieren en su perfil horario?
gl_int = mod_A["gl"]  # el modelo saturado tiene 0 g.l. residuales
p_int = stats.chi2.sf(mod_A["deviance"], gl_int)
print(f"Deviance residual del modelo multiplicativo: {mod_A['deviance']:.1f} con {gl_int} g.l. → p = {p_int:.2g}")


def empirico_bayes(observado, esperado):
    # Sobredispersión NB por momentos: Var(Y) = mu + alpha * mu²
    alpha = max(np.sum((observado - esperado) ** 2 - observado) / np.sum(esperado ** 2), 0)
    w = 1 / (1 + alpha * esperado)
    return w * esperado + (1 - w) * observado, w, alpha


panel["EB"], panel["peso_modelo"], alpha_A = empirico_bayes(panel["graves"].to_numpy(float), panel["esperado_A"].to_numpy())
print(f"alpha (sobredispersión NB) = {alpha_A:.3f} · peso medio del modelo en la estimación EB: {panel['peso_modelo'].mean():.0%}")

tasa_media_hora = panel["graves"].sum() / panel["horas"].sum()
panel["RR_celda"] = panel["EB"] / panel["horas"] / tasa_media_hora
panel["graves_anio_EB"] = panel["EB"] / N_ANIOS

In [ ]:
TOP_ZH = 15
orden_z = zonas.head(TOP_ZH).index
matriz_rr = panel.pivot(index="zona", columns="horario", values="RR_celda").loc[orden_z, HORARIOS]
matriz_g = panel.pivot(index="zona", columns="horario", values="graves").loc[orden_z, HORARIOS]
matriz_rr.columns = [textwrap.fill(c, 14) for c in matriz_rr.columns]

fig, ax = plt.subplots(figsize=(13, 0.42 * TOP_ZH + 1.6))
heatmap(matriz_rr, f"Riesgo relativo de siniestro grave por hora — {TOP_ZH} zonas con más graves (estimación EB; 1 = promedio)",
        CMAP_ROJO, ".1f", ax, "RR")
ax.axvline(4, color=TINTA, lw=1.5)
plt.tight_layout(); plt.show()

# Celdas que se apartan del perfil horario de su zona (observado >> esperado por el modelo multiplicativo)
panel["obs_sobre_esp"] = panel["graves"] / panel["esperado_A"]
lo_c, hi_c = ic_poisson(panel["graves"], panel["esperado_A"])
panel["exceso_signif"] = lo_c > 1
anomalias = (panel[panel["exceso_signif"] & (panel["graves"] >= 10)]
             .sort_values("obs_sobre_esp", ascending=False)
             [["zona", "horario", "graves", "esperado_A", "obs_sobre_esp", "RR_celda", "graves_anio_EB"]])
print("Combinaciones zona × horario con más graves de lo que predice el perfil general (IC 95% > 1, al menos 10 graves):")
anomalias.round(2).reset_index(drop=True)

### 3.5 Validación fuera de muestra

La prueba de fuego de un criterio de priorización es si **anticipa dónde van a ocurrir los graves en el futuro**. Se ajusta todo con **2018–2022** y se evalúa contra lo que efectivamente pasó en **2023–2024**, comparando cinco criterios para ordenar las 616 celdas zona × horario:

| Criterio | Descripción |
|---|---|
| Solo horario | Todas las zonas iguales (equivale a no tener mapa) |
| Solo zona | Todos los horarios iguales |
| Conteo histórico | Graves observados en cada celda (lo que haría un analista con una tabla dinámica) |
| Modelo A | Zona × horario multiplicativo |
| Modelo A + EB | Modelo combinado con lo observado en cada celda |

Métricas:
- **Deviance de Poisson** media (menor es mejor): error de la predicción de la *cantidad*.
- **Correlación de Spearman** (mayor es mejor): si el *orden* de las celdas se parece al real.
- **Captura top 10 %**: qué porcentaje de los graves de 2023–2024 ocurrió en el 10 % de celdas que el criterio marcaba como prioritarias.

In [ ]:
def panel_periodo(datos):
    p = (datos.groupby(["zona", "horario"], observed=False)["grave"].sum().rename("graves").reset_index())
    p["horario"] = p["horario"].astype(str)
    return p

entren = panel_periodo(dz[dz["anio"] <= ANIO_CORTE_VALIDACION]).rename(columns={"graves": "g_train"})
prueba = panel_periodo(dz[dz["anio"] >= ANIO_PRUEBA_DESDE]).rename(columns={"graves": "g_test"})
val = entren.merge(prueba, on=["zona", "horario"])
dias_tr = pd.date_range(f"{ANIO_DESDE}-01-01", f"{ANIO_CORTE_VALIDACION}-12-31")
dias_te = pd.date_range(f"{ANIO_PRUEBA_DESDE}-01-01", f"{ANIO_HASTA}-12-31")
h_tr = {t: ((dias_tr.dayofweek >= 5) == (t == "Fin de semana")).sum() * 6 for t in TIPOS_DIA}
h_te = {t: ((dias_te.dayofweek >= 5) == (t == "Fin de semana")).sum() * 6 for t in TIPOS_DIA}
val["h_train"] = val["horario"].str.split(" · ").str[0].map(h_tr).astype(float)
val["h_test"] = val["horario"].str.split(" · ").str[0].map(h_te).astype(float)
escala = val["h_test"] / val["h_train"]

Xv = matriz_diseno(val, {"zona": REF_ZONA, "horario": REF_HORARIO})
m_tr = ajustar_glm(Xv, val["g_train"], "poisson", offset=np.log(val["h_train"]))
mu_tr = m_tr["mu"]
eb_tr, _, _ = empirico_bayes(val["g_train"].to_numpy(float), mu_tr)
tot_h = val.groupby("horario")["g_train"].transform("sum") / val["zona"].nunique()
tot_z = val.groupby("zona")["g_train"].transform("sum") / val["h_train"].groupby(val["zona"]).transform("sum") * val["h_train"]
# Ajuste de nivel: el total predicho se calibra al total de entrenamiento escalado por horas
predicciones = {
    "Solo horario": tot_h * escala,
    "Solo zona": tot_z * escala,
    "Conteo histórico": val["g_train"] * escala,
    "Modelo A": mu_tr * escala,
    "Modelo A + EB": eb_tr * escala,
}


def deviance_media(y, mu):
    mu = np.clip(np.asarray(mu, float), 1e-6, None); y = np.asarray(y, float)
    return np.mean(2 * (np.where(y > 0, y * np.log(np.where(y > 0, y, 1) / mu), 0) - (y - mu)))


def captura(y, score, frac=0.10):
    k = int(np.ceil(len(y) * frac))
    idx = np.argsort(-np.asarray(score) + 1e-9 * np.random.default_rng(0).random(len(y)))[:k]
    return np.asarray(y)[idx].sum() / np.sum(y) * 100

y_te = val["g_test"]
res_val = pd.DataFrame({
    nombre: {"deviance media": deviance_media(y_te, p), "Spearman": stats.spearmanr(p, y_te)[0],
             "captura top 10% (%)": captura(y_te, p), "captura top 20% (%)": captura(y_te, p, 0.2)}
    for nombre, p in predicciones.items()
}).T
res_val.loc["Óptimo (oráculo)", ["captura top 10% (%)", "captura top 20% (%)"]] = [captura(y_te, y_te), captura(y_te, y_te, 0.2)]
print(f"Graves en {ANIO_PRUEBA_DESDE}-{ANIO_HASTA} (período de prueba): {y_te.sum():,}")
res_val.round(3)

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
r = res_val.drop(index="Óptimo (oráculo)")
colores = [GRIS, GRIS, AZUL_CLARO, AZUL, ROJO]
a1.barh(r.index[::-1], r["deviance media"][::-1], color=colores[::-1], height=0.6)
for i, v in enumerate(r["deviance media"][::-1]):
    a1.text(v, i, f" {v:.3f}", va="center", fontsize=8, color=TINTA2)
a1.set_title(f"Error de predicción {ANIO_PRUEBA_DESDE}-{ANIO_HASTA} (deviance, menor = mejor)"); a1.grid(axis="y", visible=False)
a2.barh(r.index[::-1], r["captura top 10% (%)"][::-1], color=colores[::-1], height=0.6)
for i, v in enumerate(r["captura top 10% (%)"][::-1]):
    a2.text(v, i, f" {v:.1f}%", va="center", fontsize=8, color=TINTA2)
a2.axvline(res_val.loc["Óptimo (oráculo)", "captura top 10% (%)"], color=TINTA2, ls="--", lw=1)
a2.text(res_val.loc["Óptimo (oráculo)", "captura top 10% (%)"], -0.6, " máximo posible", fontsize=8, color=TINTA2)
a2.set_title(f"% de graves {ANIO_PRUEBA_DESDE}-{ANIO_HASTA} en el top 10% de celdas"); a2.grid(axis="y", visible=False)
a2.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
a2.set_yticklabels([])
plt.tight_layout(); plt.show()

**Lectura del modelo A**

- **Horario.** Medido por hora, el horario con más siniestros graves en la ciudad es la **tarde de día hábil** (13,6 graves cada 100 h, RR 1,37), seguido por la tarde y la noche de fin de semana (RR 1,33 y 1,25). La madrugada de día hábil es el horario más tranquilo (RR 0,35), pero **la madrugada de fin de semana triplica a la de día hábil** (10,4 contra 3,5 graves cada 100 h).
- **Esto no contradice la Fase 3.** Allí se midió la *severidad por siniestro*: de madrugada, cada choque tiene más chances de ser grave. Acá se mide la *frecuencia*, es decir, cuántos graves ocurren por hora, que combina severidad con volumen de tránsito. Las dos miradas se complementan. El **patrullaje preventivo general** rinde más en las tardes, y los **operativos de alcohol y velocidad** rinden más en las noches y madrugadas de fin de semana, donde cada siniestro es más letal (ver modelo B).
- **Zonas.** *Near West Side* (45 graves por año, 4,0 veces la zona promedio) y *Austin* (43 por año, 3,8 veces) se separan claramente del resto. Les siguen *Near North Side* y *Loop* (~33 por año). Las 20 zonas con más graves (un cuarto de la ciudad) concentran el **53 %** de los graves. Por densidad, el *Loop* (7,6 graves por km² por año) y *West Garfield Park* (5,8) son las zonas más concentradas.
- **Perfil zona × horario.** Las zonas tienen un perfil horario parecido (sobredispersión baja, φ ≈ 1,2), aunque no idéntico (p < 0,001). Hay 14 combinaciones con un exceso significativo sobre el perfil general, por ejemplo *Loop*, *Austin* y *Humboldt Park* en la **madrugada de fin de semana**, o *North Lawndale* y *Auburn Gresham* el fin de semana. Son candidatas naturales para operativos focalizados.
- **Validación.** Con datos 2018–2022, el modelo A predice los graves de 2023–2024 con **menos de la mitad del error** que el conteo histórico celda por celda (deviance 1,15 contra 2,71). También concentra más graves futuros en su top 10 % (32,5 % contra 30,9 %, cuando el máximo posible es 38,5 %). La corrección EB casi no cambia el resultado, lo que confirma que el modelo multiplicativo simple alcanza.

**Conclusión: el modelo cumple lo que pidió el cliente.** Es simple, se explica en una oración ("riesgo de la zona × riesgo del horario") y anticipa mejor que mirar los conteos del pasado.

## 4. Modelo B — ¿Qué vuelve grave a un siniestro?

### 4.1 Planteo

Regresión logística sobre los 180 mil siniestros con zona asignada. La variable respuesta es **grave (sí/no)** y las explicativas son las que en la Fase 3 mostraron asociación con la severidad:

| Variable | Categoría de referencia |
|---|---|
| Franja horaria | Tarde (12-17) |
| Fin de semana | Día hábil |
| Iluminación | Día |
| Clima | Despejado |
| Tipo de choque (agrupado) | Giro |
| Familia de causa | Maniobras imprudentes |
| Vehículos involucrados | 2 |

El resultado se expresa en **odds ratios (OR) ajustados**: cuánto cambia la chance de que el siniestro sea grave por un factor, **manteniendo fijos los demás**. Como los graves son poco frecuentes (3,4 %), un OR se lee prácticamente como un riesgo relativo.

> Se excluyen a propósito las variables que se conocen **después** del hecho (magnitud del daño, "tipo de choque: heridos y/o grúa"), porque describen la consecuencia y no el contexto.

In [ ]:
CHOQUE_GRUPO = {
    "PEATÓN": "Peatón", "CICLISTA": "Ciclista", "EN ÁNGULO": "En ángulo", "FRONTAL": "Frontal",
    "OBJETO FIJO": "Objeto fijo", "GIRO": "Giro", "COLISIÓN TRASERA": "Colisión trasera",
    "ROCE LATERAL MISMA DIRECCIÓN": "Roce lateral", "ROCE LATERAL DIRECCIÓN OPUESTA": "Roce lateral",
    "VEHÍCULO MOTORIZADO ESTACIONADO": "Vehículo estacionado",
}
dz["choque_g"] = dz["primer_tipo_choque"].map(CHOQUE_GRUPO).fillna("Otros")
dz["vehiculos_g"] = dz["vehiculos"].clip(1, 3).map({1: "1", 2: "2", 3: "3 o más"})
dz["dia_g"] = np.where(dz["fin_de_semana"], "Fin de semana", "Hábil")

REFS_B = {
    "franja": "Tarde (12-17)", "dia_g": "Hábil", "luz_g": "Día", "clima_g": "Despejado",
    "choque_g": "Giro", "familia_causa": "Maniobras imprudentes", "vehiculos_g": "2",
}
NOMBRE_VAR = {"franja": "Franja", "dia_g": "Día", "luz_g": "Iluminación", "clima_g": "Clima",
              "choque_g": "Tipo de choque", "familia_causa": "Causa", "vehiculos_g": "Vehículos"}
XB = matriz_diseno(dz, REFS_B)
mod_B = ajustar_glm(XB, dz["grave"], "logistica")
efB = tabla_efectos(mod_B)
efB["variable"] = efB.index.str.split(" = ").str[0].map(NOMBRE_VAR)
efB["categoria"] = efB.index.str.split(" = ").str[1]
print(f"Parámetros: {XB.shape[1]} · graves por parámetro: {dz['grave'].sum() / XB.shape[1]:.0f}")

In [ ]:
t = efB.copy()
t = t.iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 0.27 * len(t) + 1.5))
yy = np.arange(len(t))
col = [ROJO if lo > 1 else (AZUL if hi < 1 else GRIS) for lo, hi in zip(t["ic_inf"], t["ic_sup"])]
ax.hlines(yy, t["ic_inf"], t["ic_sup"], color=col, lw=2)
ax.scatter(t["razon"], yy, color=col, s=35, zorder=3)
ax.axvline(1, color=TINTA2, ls="--", lw=1)
ax.set_xscale("log")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xticks([0.1, 0.2, 0.5, 1, 2, 5, 10, 20])
ax.set_yticks(yy, [f"{v}: {c}" for v, c in zip(t["variable"], t["categoria"])], fontsize=8)
# Separadores entre variables
cambios = [i for i in range(1, len(t)) if t["variable"].iloc[i] != t["variable"].iloc[i - 1]]
for c in cambios:
    ax.axhline(c - 0.5, color=GRILLA, lw=1)
ax.set_title("Odds ratio ajustado de siniestro grave (IC 95%, escala log)")
ax.set_xlabel("OR (1 = igual que la categoría de referencia)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
efB[["variable", "categoria", "razon", "ic_inf", "ic_sup", "p_valor"]].rename(columns={"razon": "OR"}).round(3).reset_index(drop=True)

### 4.2 ¿Se sostienen los hallazgos de la Fase 3 al ajustar?

En la Fase 3 cada factor se analizó por separado (**OR crudo**). Pero los factores están mezclados: de madrugada hay más alcohol, de noche hay otra composición de tipos de choque, etc. Se compara el OR crudo (modelo con una sola variable) con el **OR ajustado** (modelo completo). Si el OR ajustado se acerca a 1, el efecto "crudo" se explicaba por otras variables.

In [ ]:
crudos = []
for var, ref in REFS_B.items():
    m = ajustar_glm(matriz_diseno(dz, {var: ref}), dz["grave"], "logistica")
    crudos.append(tabla_efectos(m)["razon"])
crudos = pd.concat(crudos).rename("OR crudo")
comp = efB[["variable", "categoria"]].join(crudos).join(efB["razon"].rename("OR ajustado"))
comp["IC ajustado"] = [f"{a:.2f}–{b:.2f}" for a, b in zip(efB["ic_inf"], efB["ic_sup"])]
SELECCION = ["franja = Madrugada (0-5)", "franja = Noche (18-23)", "dia_g = Fin de semana",
             "luz_g = Noche con alumbrado", "luz_g = Noche sin alumbrado", "clima_g = Lluvia", "clima_g = Nieve / hielo",
             "choque_g = Peatón", "choque_g = Ciclista", "familia_causa = Alcohol / drogas", "familia_causa = Velocidad",
             "familia_causa = Desobediencia de señales"]
comp_sel = comp.loc[SELECCION].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 4.6))
yy = np.arange(len(comp_sel))[::-1]
for yi, (c, a) in zip(yy, zip(comp_sel["OR crudo"], comp_sel["OR ajustado"])):
    ax.annotate("", xy=(a, yi), xytext=(c, yi), arrowprops=dict(arrowstyle="->", color=GRIS, lw=1.2))
ax.scatter(comp_sel["OR crudo"], yy, color=AZUL_CLARO, s=45, zorder=3, label="OR crudo (Fase 3)")
ax.scatter(comp_sel["OR ajustado"], yy, color=ROJO, s=45, zorder=3, label="OR ajustado (modelo B)")
ax.axvline(1, color=TINTA2, ls="--", lw=1)
ax.set_xscale("log"); ax.set_xticks([0.5, 1, 2, 5, 10]); ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_yticks(yy, [f"{v}: {c}" for v, c in zip(comp_sel["variable"], comp_sel["categoria"])], fontsize=8)
ax.legend(frameon=False, loc="lower right")
ax.set_title("Del efecto crudo al efecto ajustado")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()
comp_sel.round(2)

### 4.3 Validación del modelo B

Se ajusta con 2018–2022 y se evalúa en 2023–2024:

- **AUC** (área bajo la curva ROC): probabilidad de que, tomando un siniestro grave y uno no grave al azar, el modelo le asigne más riesgo al grave. 0,5 = azar, 1 = perfecto.
- **Calibración**: si el modelo dice "5 % de chance", ¿efectivamente el 5 % de esos siniestros resulta grave?
- **Brier**: error cuadrático medio de la probabilidad (menor es mejor).

Se compara contra un modelo que solo usa el tipo de choque (la variable más fuerte de la Fase 3), para ver cuánto agregan las demás.

In [ ]:
tr, te = dz["anio"] <= ANIO_CORTE_VALIDACION, dz["anio"] >= ANIO_PRUEBA_DESDE
XB_tr, XB_te = XB[tr], XB[te]
m_B_tr = ajustar_glm(XB_tr, dz.loc[tr, "grave"], "logistica")
p_te = predecir(m_B_tr, XB_te)
X_choque = matriz_diseno(dz, {"choque_g": "Giro"})
m_ch = ajustar_glm(X_choque[tr], dz.loc[tr, "grave"], "logistica")
p_ch = predecir(m_ch, X_choque[te])
y_b = dz.loc[te, "grave"].astype(int)

val_B = pd.DataFrame({
    "Solo tipo de choque": {"AUC": roc_auc_score(y_b, p_ch), "Brier": brier_score_loss(y_b, p_ch)},
    "Modelo B completo": {"AUC": roc_auc_score(y_b, p_te), "Brier": brier_score_loss(y_b, p_te)},
    "Sin modelo (tasa media)": {"AUC": 0.5, "Brier": brier_score_loss(y_b, np.full(len(y_b), dz.loc[tr, "grave"].mean()))},
}).T
display(val_B.round(4))

cal = pd.DataFrame({"p": p_te, "y": y_b.to_numpy()})
cal["decil"] = pd.qcut(cal["p"].rank(method="first"), 10, labels=False) + 1
cal = cal.groupby("decil").agg(predicho=("p", "mean"), observado=("y", "mean"), n=("y", "size"))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8))
lim = max(cal["predicho"].max(), cal["observado"].max()) * 105
a1.plot([0, lim], [0, lim], color=GRIS, ls="--", lw=1)
a1.scatter(cal["predicho"] * 100, cal["observado"] * 100, color=ROJO, s=45, zorder=3)
a1.set_xlabel("% grave predicho"); a1.set_ylabel(f"% grave observado {ANIO_PRUEBA_DESDE}-{ANIO_HASTA}")
a1.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); a1.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
a1.set_title("Calibración por decil de riesgo")
a2.bar(cal.index, cal["observado"] * 100, color=[ROJO if d == 10 else AZUL for d in cal.index], width=0.7)
for d, v in zip(cal.index, cal["observado"] * 100):
    a2.text(d, v, f"{v:.1f}%", ha="center", va="bottom", fontsize=8, color=TINTA2)
a2.set_xticks(cal.index); a2.set_xlabel("Decil de riesgo predicho (10 = más riesgoso)")
a2.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0)); a2.set_title("% de graves observado por decil")
a2.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()
print(f"El decil 10 concentra el {cal.loc[10, 'observado'] * cal.loc[10, 'n'] / y_b.sum():.0%} de los graves de {ANIO_PRUEBA_DESDE}-{ANIO_HASTA}.")

### 4.4 Severidad ajustada por zona

¿Hay zonas donde los siniestros terminan **más graves de lo que corresponde** a su composición (tipo de choque, horario, iluminación, causa)? Para cada zona se compara:

- **O** = graves observados;
- **E** = graves esperados, sumando la probabilidad que el modelo B le asigna a cada uno de sus siniestros.

El cociente **O/E** (*razón estandarizada*, como la que se usa en epidemiología para comparar mortalidad entre hospitales) indica si la zona tiene un "plus" de severidad que no se explica por las variables del modelo: típicamente **velocidad de circulación, diseño de la vía o demora en la atención**. Es un insumo directo para auditorías de infraestructura.

In [ ]:
dz["p_grave_B"] = mod_B["mu"]
sev_z = dz.groupby("zona").agg(siniestros=("grave", "size"), O=("grave", "sum"), E=("p_grave_B", "sum"))
sev_z["O_E"] = sev_z["O"] / sev_z["E"]
sev_z["IC_inf"], sev_z["IC_sup"] = ic_poisson(sev_z["O"], sev_z["E"])
sev_z["señal"] = np.select([sev_z["IC_inf"] > 1, sev_z["IC_sup"] < 1], ["más severa", "menos severa"], "no distinguible")
print(sev_z["señal"].value_counts().to_string(), "\n")
for s in ["más severa", "menos severa"]:
    print(f"{s.capitalize()}: " + ", ".join(sev_z[sev_z["señal"] == s].sort_values("O_E", ascending=(s != "más severa")).index))

t = sev_z[sev_z["señal"] != "no distinguible"].sort_values("O_E")
fig, ax = plt.subplots(figsize=(10, 0.32 * len(t) + 1.4))
yy = np.arange(len(t))
col = [ROJO if s == "más severa" else AZUL for s in t["señal"]]
ax.hlines(yy, t["IC_inf"], t["IC_sup"], color=col, lw=2)
ax.scatter(t["O_E"], yy, color=col, s=40, zorder=3)
for yi, (v, o, e) in enumerate(zip(t["O_E"], t["O"], t["E"])):
    ax.text(t["IC_sup"].iloc[yi], yi, f"  {o:.0f} obs. / {e:.0f} esp.", va="center", fontsize=7, color=TINTA2)
ax.axvline(1, color=TINTA2, ls="--", lw=1)
ax.set_yticks(yy, t.index, fontsize=8)
ax.set_xlabel("Graves observados / esperados por el modelo B")
ax.set_title("Zonas con severidad significativamente distinta a la esperable (IC 95%)")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Lectura del modelo B**

- **Quién está involucrado sigue siendo lo que más pesa.** Un atropello a peatón tiene **9,9 veces** más chances de ser grave que un choque en giro, y uno a ciclista, 5 veces. Le siguen el choque frontal (2,1), los siniestros con 3 o más vehículos (2,8) y el choque contra objeto fijo (1,8).
- **Las causas de fiscalización son las más severas, incluso al ajustar.** Alcohol/drogas (OR 2,9), velocidad (1,7) y desobediencia de señales (1,5) mantienen su efecto con el horario, la iluminación y el tipo de choque fijos. El efecto de la velocidad incluso **aumenta** al ajustar (de 1,57 a 1,74). Esto refuerza la recomendación de la Fase 3: la fiscalización ataca los siniestros que más daño hacen.
- **La madrugada sigue siendo más riesgosa (OR 1,60) y el fin de semana también (1,17), aun descontando el alcohol y el tipo de choque.** Quedan factores no medidos (velocidad real de circulación con calles vacías, cansancio, demora en la atención) que justifican operativos en esas franjas.
- **La "noche" como iluminación pierde su efecto al ajustar** (OR crudo 1,43 → ajustado 1,05, no significativo). Lo que pesa es la *franja horaria* (quién circula y cómo), no la falta de luz en sí. Esto es coherente con la Fase 3, que no encontró diferencias entre noche con y sin alumbrado. **No implica** que la iluminación no sirva en puntos específicos (por ejemplo, cruces peatonales), sino que no explica la severidad a escala de ciudad.
- **Lluvia y nieve reducen la severidad** (OR 0,80 y 0,59). La hipótesis del cliente "más graves de noche con lluvia" no se sostiene como factor de severidad: se circula más despacio. Sin datos de tránsito no se puede saber si aumentan la *cantidad* de choques.
- **Validación.** AUC de 0,76 fuera de muestra, contra 0,72 si solo se usa el tipo de choque. Está bien calibrado y el 10 % de siniestros con mayor riesgo predicho concentra el **39 % de los graves** de 2023–2024. En el decil más alto sobreestima levemente (14,5 % predicho contra 12,4 % observado), algo esperable si 2024 tiene graves todavía sin registrar (limitación 5 de la Fase 3).
- **Severidad ajustada por zona: dos ciudades distintas.** Las zonas céntricas y del norte costero (*Near North Side*, *Lincoln Park*, *Uptown*, *Lake View*, *Loop*, *Near West Side*, *West Town*) tienen **siniestros menos graves de lo esperable**: muchos choques, pero a baja velocidad por la congestión. En cambio, varias zonas del **West Side** (*North Lawndale*, *East* y *West Garfield Park*) y del **South Side** (*West Englewood*, *Auburn Gresham*, *Ashburn*, *Woodlawn*, *Calumet Heights*, *West Pullman*, *Pullman*) tienen entre un 20 % y un 80 % **más graves de lo que corresponde a su composición**. La hipótesis a verificar es que haya velocidades de circulación más altas en avenidas anchas, y es el insumo más directo para las auditorías de infraestructura y los radares.

## 5. Índice de priorización por zona

Se integra todo en una **ficha por zona** que el Observatorio pueda leer sin conocer los modelos:

| Columna | Origen | Cómo se lee |
|---|---|---|
| `graves_por_anio` | Datos | Cuánto daño concentra la zona (criterio principal de prioridad) |
| `RR_zona` | Modelo A | Veces el promedio de una zona de Chicago |
| `graves_km2_anio` | Datos + límites | Densidad: separa zonas grandes de zonas realmente concentradas |
| `O_E_severidad` | Modelo B | > 1: los siniestros terminan más graves de lo esperable → revisar velocidad / diseño |
| `horario_critico` | Modelo A + EB | Horario con más graves por hora en esa zona → asignación de patrullaje |
| `horario_con_exceso` | Modelo A | Horario en el que la zona tiene **más graves de lo que predice el perfil general** de la ciudad (IC 95 % > 1). "—" si no hay ninguno → operativos focalizados |
| `%_vulnerables` | Datos | % de los graves que son peatones o ciclistas → infraestructura peatonal |
| `palanca_dominante` | Fase 3 | Palanca con más graves entre las causas determinadas |

In [ ]:
graves_dz = dz[dz["grave"]]
vuln = graves_dz.groupby("zona")["choque_g"].apply(lambda s: s.isin(["Peatón", "Ciclista"]).mean() * 100)
det = graves_dz[~graves_dz["palanca"].isin(["Sin determinar", "Otros"])]
pal_dom = det.groupby("zona")["palanca"].agg(lambda s: s.value_counts().index[0])
pal_pct = det.groupby("zona")["palanca"].agg(lambda s: s.value_counts(normalize=True).iloc[0] * 100)
idx_h = panel.loc[panel.groupby("zona")["RR_celda"].idxmax()].set_index("zona")
exceso_z = (panel[panel["exceso_signif"] & (panel["graves"] >= 10)].sort_values("obs_sobre_esp", ascending=False)
            .drop_duplicates("zona").set_index("zona"))
exceso_z = exceso_z["horario"] + " (" + exceso_z["obs_sobre_esp"].map("{:.1f}×".format) + ")"
nocturno = graves_dz.groupby("zona")["luz_g"].apply(lambda s: s.isin(["Noche con alumbrado", "Noche sin alumbrado"]).mean() * 100)

ficha = zonas[["graves_por_anio", "RR_zona", "RR_IC_inf", "RR_IC_sup", "graves_km2_anio", "tasa_graves_%", "siniestros"]].join([
    sev_z[["O_E", "señal"]].rename(columns={"O_E": "O_E_severidad", "señal": "severidad_ajustada"}),
    idx_h[["horario", "RR_celda"]].rename(columns={"horario": "horario_critico", "RR_celda": "RR_horario_critico"}),
    exceso_z.rename("horario_con_exceso"), vuln.rename("%_vulnerables"), nocturno.rename("%_graves_nocturnos"),
    pal_dom.rename("palanca_dominante"), pal_pct.rename("%_palanca_dominante"),
])
ficha["horario_con_exceso"] = ficha["horario_con_exceso"].fillna("—")
ficha["con_heridos_por_anio"] = zonas["con_heridos"] / N_ANIOS
ficha.insert(0, "prioridad", ficha["graves_por_anio"].rank(ascending=False, method="first").astype(int))
ficha = ficha.sort_values("prioridad")
ficha.head(15)[["prioridad", "graves_por_anio", "RR_zona", "graves_km2_anio", "O_E_severidad", "severidad_ajustada",
                "horario_critico", "horario_con_exceso", "%_vulnerables", "palanca_dominante"]].round(2)

### 5.1 Mapa de riesgo relativo por zona

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 8))
for ax, col, titulo, cmap, norm_log in [
    (a1, "RR_zona", "Riesgo relativo de siniestros graves\n(1 = zona promedio)", CMAP_ROJO, True),
    (a2, "O_E_severidad", "Severidad ajustada (observado / esperado)\nrojo: más severa de lo esperable", CMAP_DIV, False),
]:
    parches, valores = [], []
    for f in areas_geo["features"]:
        nombre = f["properties"]["community"].title().replace("Ohare", "O'Hare")
        v = ficha.loc[nombre, col] if nombre in ficha.index else np.nan
        for poligono in f["geometry"]["coordinates"]:
            parches.append(Polygon(np.asarray(poligono[0]), closed=True)); valores.append(v)
    valores = np.asarray(valores, float)
    if norm_log:
        norm = plt.matplotlib.colors.LogNorm(vmin=np.nanmin(valores), vmax=np.nanmax(valores))
    else:
        dev = np.nanmax(np.abs(np.log(valores)))
        norm = plt.matplotlib.colors.LogNorm(vmin=np.exp(-dev), vmax=np.exp(dev))
    pc = PatchCollection(parches, cmap=cmap, norm=norm, edgecolor="white", linewidth=0.5)
    pc.set_array(valores)
    ax.add_collection(pc)
    ax.autoscale_view(); ax.set_aspect(1 / np.cos(np.radians(41.84)))
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    cb = fig.colorbar(pc, ax=ax, shrink=0.5)
    ticks = [t for t in [0.1, 0.25, 0.5, 0.75, 1, 1.5, 2, 4] if norm.vmin <= t <= norm.vmax]
    cb.set_ticks(ticks); cb.set_ticklabels([f"{t:g}" for t in ticks]); cb.ax.minorticks_off()
    ax.set_title(titulo)
# Etiquetas de las 10 zonas prioritarias
centroides = dz.groupby("zona")[["longitud", "latitud"]].median()
for z in ficha.head(10).index:
    a1.annotate(str(ficha.loc[z, "prioridad"]), centroides.loc[z], fontsize=8, ha="center", va="center", color=TINTA,
                fontweight="bold", bbox=dict(boxstyle="circle,pad=0.2", fc="white", ec=TINTA2, lw=0.6))
a1.text(0.0, 0.0, "\n".join(f"{p}. {z}" for z, p in ficha.head(10)["prioridad"].items()), transform=a1.transAxes,
        fontsize=8, color=TINTA, va="bottom", bbox=dict(fc="white", ec=GRILLA))
plt.tight_layout(); plt.show()

Versión interactiva (se guarda como `mapa_riesgo_zonas.html` y se abre en cualquier navegador): al pasar el mouse sobre cada zona se ve su ficha completa.

In [ ]:
import branca.colormap as bcm
import folium

geo_ficha = json.loads(json.dumps(areas_geo))
for f in geo_ficha["features"]:
    z = f["properties"]["community"].title().replace("Ohare", "O'Hare")
    r = ficha.loc[z]
    f["properties"].update({
        "zona": z, "prioridad": int(r["prioridad"]), "graves_anio": f"{r['graves_por_anio']:.1f}",
        "rr": float(r["RR_zona"]), "rr_txt": f"{r['RR_zona']:.2f}", "oe": f"{r['O_E_severidad']:.2f} ({r['severidad_ajustada']})",
        "horario": f"{r['horario_critico']} (RR {r['RR_horario_critico']:.1f})", "exceso": r["horario_con_exceso"],
        "vuln": f"{r['%_vulnerables']:.0f}%", "palanca": f"{r['palanca_dominante']} ({r['%_palanca_dominante']:.0f}%)",
    })
escala_rr = bcm.StepColormap(["#f4f8fd", "#86b6ef", "#f2a3a2", "#e34948", "#8f1d1c"], index=[0, 0.5, 1, 1.5, 2.5, 6],
                             vmin=0, vmax=6, caption="Riesgo relativo de siniestros graves (1 = zona promedio)")
mapa = folium.Map(location=[41.84, -87.68], zoom_start=11, tiles="cartodbpositron")
folium.GeoJson(
    geo_ficha, name="Riesgo relativo por zona",
    style_function=lambda f: {"fillColor": escala_rr(min(f["properties"]["rr"], 5.9)), "color": "white",
                              "weight": 1, "fillOpacity": 0.75},
    highlight_function=lambda f: {"weight": 3, "color": TINTA},
    tooltip=folium.GeoJsonTooltip(
        fields=["zona", "prioridad", "graves_anio", "rr_txt", "oe", "horario", "exceso", "vuln", "palanca"],
        aliases=["Zona", "Prioridad", "Graves por año", "Riesgo relativo", "Severidad ajustada (O/E)",
                 "Horario crítico", "Horario con exceso propio", "% graves peatón/ciclista", "Palanca dominante"]),
).add_to(mapa)
escala_rr.add_to(mapa)
mapa.save("mapa_riesgo_zonas.html")
mapa

## 6. Puntos críticos corregidos por regresión a la media (Empírico Bayesiano)

El ranking de la Fase 3 ordena las celdas de ~150 m por un índice de severidad calculado con los conteos **observados**. Eso tiene un problema conocido en seguridad vial: la **regresión a la media**. Una esquina que tuvo, por azar, varios graves en el período va a aparecer arriba del ranking aunque su riesgo real sea normal, y "mejorará sola" el año siguiente, intervenga o no la ciudad (lo que además haría parecer exitosa una intervención que no lo fue).

El método **Empírico Bayesiano** (recomendado por el *Highway Safety Manual* de AASHTO) corrige esto estimando para cada celda:

$$\text{graves}_{EB} = w \cdot \underbrace{\mu}_{\text{esperado por su contexto}} + (1 - w) \cdot \underbrace{K}_{\text{observado}}, \qquad w = \frac{1}{1 + \alpha\,\mu}$$

- $\mu$ = graves esperados en la celda según **su volumen de siniestros y la tasa de severidad (ajustada) de su zona**: $\mu = \sum_i p_i$, la suma de las probabilidades del modelo B para los siniestros de la celda;
- $K$ = graves observados;
- $\alpha$ = sobredispersión, estimada con los datos. Cuanto más varía la severidad real entre celdas, más peso recibe lo observado.

Sin datos de flujo vehicular, la cantidad total de siniestros de la celda funciona como *proxy* de exposición. La sección 6.3 contrasta ese proxy con el aforo real que publica el portal de Chicago.

In [ ]:
CELDA = 0.0015  # igual que la Fase 3
dz["celda"] = dz["celda_150m"]
dz["via"] = ((dz["direccion_calle"].fillna("") + " " + dz["calle"].fillna("")).str.strip().str.title()
             .str.replace(r"(\d)(St|Nd|Rd|Th)\b", lambda m: m[1] + m[2].lower(), regex=True))
dz["peso_epdo"] = np.select([dz["fatal"], dz["grave"], dz["con_heridos"]], [10, 5, 1], default=0)


def nombre_lugar(vias):
    top = vias[vias != ""].value_counts().index[:2]
    return " / ".join(top)


def celdas_eb(datos, prob):
    c = datos.assign(p=prob).groupby("celda").agg(
        siniestros=("grave", "size"), K=("grave", "sum"), fatales=("fatal", "sum"), mu=("p", "sum"),
        epdo=("peso_epdo", "sum"))
    c["EB"], c["w"], alpha = empirico_bayes(c["K"].to_numpy(float), c["mu"].to_numpy())
    return c, alpha


cel, alpha_cel = celdas_eb(dz, dz["p_grave_B"])
extra = dz.groupby("celda").agg(lugar=("via", nombre_lugar), zona=("zona", lambda s: s.mode().iat[0]),
                                lat=("latitud", "mean"), lon=("longitud", "mean"),
                                pct_nocturno=("luz_g", lambda s: s.isin(["Noche con alumbrado", "Noche sin alumbrado"]).mean() * 100),
                                pct_vulnerables=("choque_g", lambda s: s.isin(["Peatón", "Ciclista"]).mean() * 100))
cel = cel.join(extra)
cel["exceso_EB"] = cel["EB"] - cel["mu"]  # graves por encima de lo esperable por su volumen y contexto
cel["graves_anio_EB"] = cel["EB"] / N_ANIOS
cel["rank_EB"] = cel["EB"].rank(ascending=False, method="first").astype(int)
cel["rank_fase3"] = cel["epdo"].rank(ascending=False, method="first").astype(int)
print(f"Celdas: {len(cel):,} · alpha = {alpha_cel:.3f} · peso medio del 'esperado' en celdas con ≥ 5 graves: "
      f"{cel.loc[cel['K'] >= 5, 'w'].mean():.0%}")

### 6.1 Validación: ¿qué ranking anticipa mejor los graves futuros?

Se arma cada ranking con 2018–2022 y se cuenta cuántos graves ocurrieron en 2023–2024 en las **50 celdas** que cada uno marcaba como prioritarias (el orden de magnitud de un programa anual de intervenciones).

In [ ]:
tr_d, te_d = dz[dz["anio"] <= ANIO_CORTE_VALIDACION], dz[dz["anio"] >= ANIO_PRUEBA_DESDE]
m_tr_B = ajustar_glm(XB[dz["anio"] <= ANIO_CORTE_VALIDACION], tr_d["grave"], "logistica")
cel_tr, _ = celdas_eb(tr_d, m_tr_B["mu"])
cel_tr = cel_tr.join(te_d.groupby("celda")["grave"].sum().rename("K_test")).fillna({"K_test": 0})
total_test = te_d["grave"].sum()

criterios = {"Graves observados": "K", "Índice EPDO (Fase 3)": "epdo", "Total de siniestros": "siniestros", "Empírico Bayesiano": "EB"}
filas = []
for N_TOP in [25, 50, 100]:
    fila = {"top": N_TOP}
    for nombre, col in criterios.items():
        orden = cel_tr.sort_values([col, "siniestros"], ascending=False).head(N_TOP)
        fila[nombre] = orden["K_test"].sum()
    fila["Óptimo (oráculo)"] = cel_tr["K_test"].nlargest(N_TOP).sum()
    filas.append(fila)
val_cel = pd.DataFrame(filas).set_index("top")
display(val_cel.astype(int))
print(f"Graves totales en {ANIO_PRUEBA_DESDE}-{ANIO_HASTA}: {total_test:,}")

fig, ax = plt.subplots(figsize=(10, 3.4))
v = val_cel.loc[50].drop("Óptimo (oráculo)").sort_values()
ax.barh(v.index, v.values, color=[ROJO if i == "Empírico Bayesiano" else AZUL_CLARO for i in v.index], height=0.6)
for i, x in enumerate(v.values):
    ax.text(x, i, f" {x:.0f} graves", va="center", fontsize=8, color=TINTA2)
ax.set_title(f"Graves de {ANIO_PRUEBA_DESDE}-{ANIO_HASTA} ocurridos en las 50 celdas priorizadas con datos {ANIO_DESDE}-{ANIO_CORTE_VALIDACION}")
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

### 6.2 Ranking corregido de puntos críticos

Ranking final con los datos completos 2018–2024, comparado con la posición que cada punto tenía en la Fase 3.

In [ ]:
TOP_N = 20
top_eb = cel.sort_values("EB", ascending=False).head(TOP_N)
tabla_top = top_eb[["rank_EB", "rank_fase3", "lugar", "zona", "siniestros", "K", "fatales", "mu", "EB",
                    "graves_anio_EB", "pct_nocturno", "pct_vulnerables"]].rename(columns={
    "rank_EB": "ranking", "rank_fase3": "ranking Fase 3", "K": "graves obs.", "mu": "graves esperados",
    "EB": "graves EB", "graves_anio_EB": "graves EB/año", "pct_nocturno": "% nocturno", "pct_vulnerables": "% peatón/ciclista"})
display(tabla_top.round(1).reset_index(drop=True))

top_f3 = cel.nsmallest(15, "rank_fase3")
siguen = top_f3["rank_EB"].le(TOP_N).sum()
print(f"De los 15 puntos del top de la Fase 3, {siguen} siguen en el top {TOP_N} corregido.")
print("Puntos del top 15 de la Fase 3 que caen al corregir:")
display(top_f3.loc[top_f3["rank_EB"] > TOP_N, ["rank_fase3", "rank_EB", "lugar", "siniestros", "K", "mu", "EB"]].round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 0.42 * TOP_N + 1.2))
t = top_eb.iloc[::-1]
etq = [f"#{r} {l}" for r, l in zip(t["rank_EB"], t["lugar"])]
yy = np.arange(len(t))
ax.scatter(t["K"], yy, color=AZUL_CLARO, s=45, label="Graves observados", zorder=3)
ax.scatter(t["mu"], yy, color=GRIS, marker="|", s=150, label="Esperado por volumen y contexto", zorder=3)
ax.scatter(t["EB"], yy, color=ROJO, s=55, label="Estimación EB (para priorizar)", zorder=4)
for yi, (k, e) in enumerate(zip(t["K"], t["EB"])):
    ax.plot([min(k, e), max(k, e)], [yi, yi], color=GRIS, lw=1, zorder=2)
ax.set_yticks(yy, etq, fontsize=8)
ax.set_xlabel(f"Siniestros graves {ANIO_DESDE}-{ANIO_HASTA}")
ax.set_title(f"Top {TOP_N} puntos críticos corregidos por regresión a la media")
ax.legend(frameon=False, loc="lower right", fontsize=8)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

**Lectura de la corrección Empírico Bayesiana**

- **La severidad "propia" de cada cruce varía poco más allá del azar** (α bajo). El mejor predictor de cuántos graves tendrá un cruce en el futuro es **cuántos siniestros tiene y de qué tipo**, más que cuántos graves tuvo. Por eso el método EB lleva las estimaciones hacia lo esperado por volumen y contexto.
- **Ordenar por graves observados es el peor criterio.** Las 50 celdas con más graves en 2018–2022 tuvieron 64 graves en 2023–2024. Las 50 elegidas con EB tuvieron **89 (+39 %)**. Es la regresión a la media en acción. El índice EPDO de la Fase 3 (87) y el total de siniestros (90) rinden casi igual que EB, porque también usan la información de los choques no graves.
- **Se recomienda adoptar EB como criterio oficial.** Rinde igual que la mejor alternativa, tiene un respaldo técnico estándar (*Highway Safety Manual*) que se puede defender ante el Concejo y, sobre todo, es el método que se va a necesitar para **evaluar a los 12 meses si las intervenciones funcionaron** sin confundir el efecto de la obra con la regresión a la media.
- **El top de la Fase 3 se mantiene en lo esencial:** 12 de los 15 puntos siguen en el top 20 corregido, y *E Lower Wacker Dr / N Columbus Dr* sigue primero. Hay tres casos para comentar:
  - *N Pulaski Rd / W Madison St* tuvo 18 graves en solo 145 siniestros (el caso más llamativo de la Fase 3). La estimación EB es de ~9: **sigue en el top (#6)**, pero parte de ese exceso fue probablemente azar.
  - *N Halsted St / W Hubbard St* sube al #3 con solo 6 graves observados, porque tiene 335 siniestros. Es un punto de mucho volumen donde estadísticamente "tocaría" tener más graves.
  - *S Western Ave / W 47th St* entra al top (#15) con solo 3 graves observados, impulsado únicamente por su volumen. Es el tipo de punto que conviene **verificar con el Observatorio** antes de incluirlo en una lista de obra.

### 6.3 ¿Cambia el ranking si se mide la exposición real?

El ranking EB supone que una celda con más siniestros tiene más tránsito. Es una suposición fuerte: un cruce de paso con mucho tráfico lento puede acumular choques leves y quedar arriba sin ser más peligroso por vehículo. El portal de datos abiertos de Chicago publica el **AADT** (tránsito diario promedio) de 2.008 segmentos medidos por las cámaras del CDOT entre 2022 y 2024, así que la suposición se puede contrastar con una medición.

Se comparan cuatro formas de repartir la exposición esperada entre las celdas con siniestros:

| Proxy | Qué mide | Fuente |
|---|---|---|
| **Modelo B (oficial)** | Graves esperados según la composición de cada siniestro y la severidad de su zona | Sección 5 |
| **Total de siniestros** | Volumen de choques de la celda | Fase 3 |
| **EPDO** | Volumen de choques ponderado por severidad | Fase 3 |
| **AADT medido** | Tránsito diario del segmento más cercano de la misma calle | Portal de Chicago |

Los cuatro reparten la misma cantidad de graves observados, así que las estimaciones son comparables: lo único que cambia es **cómo se reparte lo esperado**. Donde no hay aforo medido se imputa el AADT mediano de la ciudad; es una imputación neutra y no una medición, así que la validación se repite sobre el soporte con aforo suficiente.


In [ ]:
from aforos import asignar_aforo, asignar_aforo_por_via, cargar_aforos, normalizar_via

# El portal publica el aforo del punto medio de cada segmento, no el de cada cuadra, y nombra las
# calles sin prefijo ni sufijo (`WESTERN`, `79TH`). El parte de siniestros las nombra al revés
# (`N WESTERN AVE`, `E 79TH ST`), así que hay que normalizar los dos lados antes de emparejar.
aforos = cargar_aforos()
dz[["aforo", "aforo_distancia_m"]] = asignar_aforo_por_via(
    dz["latitud"], dz["longitud"], dz["calle"], aforos
)
expo = dz.groupby("celda").agg(con_aforo=("aforo", "count"), aadt=("aforo", "mean"))
expo["pct_aforado"] = expo["con_aforo"] / dz.groupby("celda").size()
sin_nombre = asignar_aforo(dz["latitud"], dz["longitud"], aforos)  # línea base: solo radio

top20_eb = cel["rank_EB"].le(20)
cobertura = pd.Series({
    "Siniestros con aforo (misma vía)": dz["aforo"].notna().mean() * 100,
    "Siniestros con aforo (solo radio de 75 m)": sin_nombre["aforo"].notna().mean() * 100,
    "Graves con aforo (misma vía)": dz.loc[dz["grave"], "aforo"].notna().mean() * 100,
    "Celdas con algún siniestro aforado": expo["pct_aforado"].gt(0).mean() * 100,
    "Celdas con ≥ 50 % aforado": expo["pct_aforado"].ge(0.5).mean() * 100,
    "Top 20 del EB con ≥ 50 % aforado": cel.loc[top20_eb].join(expo)["pct_aforado"].ge(0.5).mean() * 100,
}, name="%")
print(f"Segmentos: {len(aforos):,} · vías: {normalizar_via(aforos['roadname']).nunique()} · "
      f"AADT mediano: {aforos['aforo'].median():,.0f} veh/día")
print(f"Distancia mediana al segmento asignado: {dz['aforo_distancia_m'].median():,.0f} m")
display(cobertura.round(1).to_frame())

puntos_top = cel.loc[top20_eb].join(expo)[["lugar", "siniestros", "K", "pct_aforado", "aadt"]]
display(puntos_top.round(2).reset_index(drop=True))


In [ ]:
MEDIANA_AADT = aforos["aforo"].median()
expo["aadt_imputado"] = expo["aadt"].fillna(MEDIANA_AADT)
expuesto = cel.join(expo)
K_cel = expuesto["K"].to_numpy(float)
total_K = K_cel.sum()

PROXIES = {
    "Modelo B (oficial)": expuesto["mu"],
    "Total de siniestros": total_K * expuesto["siniestros"] / expuesto["siniestros"].sum(),
    "EPDO (Fase 3)": total_K * expuesto["epdo"] / expuesto["epdo"].sum(),
    "AADT medido": total_K * expuesto["aadt_imputado"] / expuesto["aadt_imputado"].sum(),
}
EB_proxy = {}
for nombre, mu in PROXIES.items():
    EB_proxy[nombre], _, _ = empirico_bayes(K_cel, mu.to_numpy(float))
EB_proxy = pd.DataFrame(EB_proxy, index=expuesto.index)
ranks = EB_proxy.rank(ascending=False, method="first")

# Estas cuatro columnas viajan al CSV de puntos críticos (sección 8).
cel["rank_conteo"] = ranks["Total de siniestros"].astype(int)
cel["rank_epdo"] = ranks["EPDO (Fase 3)"].astype(int)
cel["rank_aadt"] = ranks["AADT medido"].astype(int)
cel["aadt"] = expuesto["aadt"]
cel["pct_aforado"] = expuesto["pct_aforado"]
apariciones = pd.concat({n: ranks[n].le(20) for n in ranks}, axis=1).sum(axis=1)
cel["estable_top20"] = np.where(apariciones.eq(ranks.shape[1]), "sí", "no")

top20 = {n: set(ranks.nsmallest(20, n).index) for n in ranks}
coincidencia = pd.DataFrame({a: {b: len(top20[a] & top20[b]) for b in top20} for a in top20})
print("Puntos del top 20 en común entre proxies:")
display(coincidencia)
print("Correlación de Spearman entre las estimaciones EB (todas las celdas):")
display(EB_proxy.corr(method="spearman").round(2))
print(f"El top 20 oficial comparte {len(top20['Modelo B (oficial)'] & top20['AADT medido'])} puntos con el AADT "
      f"y {len(top20['Modelo B (oficial)'] & top20['Total de siniestros'])} con el total de siniestros.")
print(f"Celdas en el top 20 con los cuatro proxies: {int(apariciones.eq(ranks.shape[1]).sum())} "
      f"({int(apariciones[top20_eb].eq(ranks.shape[1]).sum())} de las 20 del ranking oficial).")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.4 * 20 + 1.2))
t = cel.loc[top20_eb].sort_values("rank_EB", ascending=False)
yy = np.arange(len(t))
ax.hlines(yy, t["rank_EB"], t["rank_aadt"], color=GRILLA, lw=2, zorder=1)
ax.scatter(t["rank_EB"], yy, color=ROJO, s=45, zorder=3, label="Ranking oficial (Modelo B)")
ax.scatter(t["rank_aadt"], yy, color=AZUL, s=45, zorder=3, label="Ranking con AADT medido")
ax.set_xscale("log")
ax.set_yticks(yy, [f"{l} ({p:.0%} aforado)" for l, p in zip(t["lugar"], t["pct_aforado"])], fontsize=7)
ax.set_xlabel("Posición en el ranking (escala logarítmica; más a la izquierda = más prioritario)")
ax.set_title("Cómo se mueve el top 20 cuando se mide la exposición")
ax.legend(frameon=False, loc="lower right", fontsize=8)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()


In [ ]:
# La validación fuera de muestra de la sección 6.1 se repite con cada proxy: se entrena hasta
# ANIO_CORTE_VALIDACION y se cuentan los graves de ANIO_PRUEBA_DESDE a ANIO_HASTA que capturan las
# 50 celdas prioritarias.
val_proxy_cel = cel_tr.join(expo)
val_proxy_cel["aadt_imputado"] = val_proxy_cel["aadt"].fillna(MEDIANA_AADT)
K_tr = val_proxy_cel["K"].to_numpy(float)
total_tr = K_tr.sum()
proxies_val = {
    "Modelo B (oficial)": val_proxy_cel["mu"],
    "Total de siniestros": total_tr * val_proxy_cel["siniestros"] / val_proxy_cel["siniestros"].sum(),
    "EPDO (Fase 3)": total_tr * val_proxy_cel["epdo"] / val_proxy_cel["epdo"].sum(),
    "AADT medido": total_tr * val_proxy_cel["aadt_imputado"] / val_proxy_cel["aadt_imputado"].sum(),
}
filas = []
for nombre, mu in proxies_val.items():
    eb, _, _ = empirico_bayes(K_tr, mu.to_numpy(float))
    prioridad = pd.Series(eb, index=val_proxy_cel.index)
    fila = {"proxy": nombre}
    for columna, datos in [("todas las celdas", val_proxy_cel),
                           ("soporte con aforo", val_proxy_cel[val_proxy_cel["pct_aforado"].ge(0.5)])]:
        elegidas = prioridad.loc[datos.index].nlargest(50).index
        fila[columna] = int(datos.loc[elegidas, "K_test"].sum())
    filas.append(fila)
val_proxy = pd.DataFrame(filas).set_index("proxy")
display(val_proxy)
print(f"Graves totales de {ANIO_PRUEBA_DESDE}-{ANIO_HASTA} en la base: {int(total_test):,}")

margen = val_proxy.loc["AADT medido"] / val_proxy.loc["Modelo B (oficial)"] - 1
print(f"El AADT captura {margen['todas las celdas']:+.0%} más graves futuros que el proxy oficial "
      f"(y {margen['soporte con aforo']:+.0%} sobre el soporte con aforo).")
CLARO = margen["todas las celdas"] >= 0.10 and margen["soporte con aforo"] >= 0.10
print("Decisión: " + (
    "el AADT pasa a ser el proxy oficial del ranking EB."
    if CLARO else
    "se mantiene el Modelo B como proxy oficial. El AADT se informa como contraste en "
    "`puntos_criticos_eb.csv` y en `validacion_observatorio.csv`, porque un margen menor al 10 % "
    "no alcanza para reemplazar un modelo que además explica la severidad."
))


**Lectura del contraste**

- **El ranking no es el mismo según cómo se mida la exposición.** El AADT comparte 8 de los 20 puntos con el ranking oficial y su correlación de Spearman con el EB es de 0,43. Los tres proxies basados en choques (Modelo B, total de siniestros y EPDO) coinciden entre sí: 14 o 15 de 20 puntos en común y Spearman de 0,84 a 0,92.
- **El AADT premia los corredores de mucho tránsito y castiga los puntos de volumen medio.** *N Pulaski Rd / W Augusta Blvd* pasa del puesto 245 al 6, *S Cicero Ave / W Lexington St* del 14 al 4 y *S Halsted St / W 99th St* del 42 al 5. En el otro extremo, *N Halsted St / W Hubbard St* cae del 3 al 105, *S Western Ave / W 47th St* del 15 al 366 y *W Taylor St / S Desplaines St* del 8 al 97.
- **La cobertura no alcanza para reemplazar el modelo.** El aforo cubre el 59,3 % de los siniestros (12,9 % si se busca solo por cercanía), el 39,5 % de las celdas tiene al menos la mitad de sus siniestros aforados y 4 de los 20 puntos del top oficial no llegan a esa mitad; en uno de ellos ningún siniestro tiene aforo.
- **El AADT no anticipa mejor los graves futuros.** En la validación 2018-2022 → 2023-2024, el top 50 por AADT captura 59 de los 1.620 graves de 2023-2024, contra 89 del proxy oficial (-34 %). Restringido al soporte con aforo el resultado es el mismo: 43 contra 73 (-41 %). Medir la exposición con más precisión no mejora la predicción del ranking y sí la empeora.
- **Decisión: se mantiene el Modelo B como proxy oficial.** El AADT queda como contraste en `puntos_criticos_eb.csv` (`rank_aadt`) y como criterio de verificación en `validacion_observatorio.csv`: 10 de los 20 puntos quedan marcados como `verificacion_prioritaria = sí`, y solo 6 se sostienen en el top 20 con los cuatro proxies.
- **Qué significa para el Observatorio.** Los puntos marcados son los que hay que confirmar primero en terreno: su prioridad se apoya en el volumen de choques y no en la exposición medida. *S Western Ave / W 47th St* (3 graves observados, 272 siniestros, 10.688 veh/día) es el caso más claro: los tres proxies de choques lo suben al top y el AADT lo deja en el puesto 366.

La exposición sigue siendo la limitación principal del modelo (sección 9), pero ahora está **medida y acotada**: se sabe cuánto puede mover el ranking y en qué puntos.


## 7. Validación con el Observatorio Vial

La validación estadística (secciones 3.5, 4.3 y 6.1) dice si el modelo **anticipa** los graves. La validación con el Observatorio dice si el resultado **tiene sentido en el terreno** y si hay información que la base no captura (obras ya realizadas, cambios de sentido de circulación, cámaras instaladas, errores de geocodificación). Ambas son necesarias antes de presentar el listado ante el Concejo.

### 7.1 Protocolo propuesto

| # | Qué se valida | Pregunta concreta al Observatorio | Si la respuesta es "no" |
|---|---|---|---|
| 1 | **Definición de zona** | ¿Las áreas comunitarias sirven para planificar, o el patrullaje se organiza por distritos policiales / *wards*? | Reagregar el modelo A a la unidad que corresponda (cambia una sola línea: el polígono de asignación) |
| 2 | **Definición de horario** | ¿Los turnos de patrullaje coinciden con franjas de 6 h? | Redefinir las franjas en `preparacion.py` |
| 3 | **Criterio de severidad** | ¿KSI (fatal + incapacitante) es el criterio que quiere usar el Concejo? | Reajustar con *con heridos* o con el índice EPDO como variable respuesta |
| 4 | **Top de zonas** | ¿Las 10 zonas prioritarias coinciden con su experiencia? ¿Alguna sorprende? | Revisar la carga de esa zona (posible error de geocodificación o de clasificación) |
| 5 | **Top de puntos críticos** | Para cada punto del top 20: ¿existe esa intersección? ¿ya hubo intervención entre 2018 y hoy? | Marcar como "intervenido" y excluir de la priorización (o usarlo como caso de evaluación antes/después) |
| 6 | **Zonas con severidad ajustada alta** | ¿Hay corredores de alta velocidad, obras o problemas de diseño conocidos en esas zonas? | Documentar como hipótesis a verificar en la auditoría de seguridad vial |
| 7 | **Palancas** | ¿El agrupamiento causa → palanca de la Fase 3 refleja cómo se asigna el presupuesto? | Reasignar familias en `preparacion.py` |

### 7.2 Planilla de validación

Se genera `validacion_observatorio.csv` con las 10 zonas y los 20 puntos prioritarios, con columnas vacías para que el Observatorio complete (¿confirmado?, intervención existente, comentarios). Al recibirla, las correcciones se incorporan y se vuelve a correr este notebook.

In [ ]:
fz = ficha.head(10).reset_index().rename(columns={"index": "zona"})
plantilla_z = pd.DataFrame({
    "tipo": "Zona", "prioridad": fz["prioridad"], "lugar": fz["zona"], "graves_por_anio": fz["graves_por_anio"],
    "ubicacion": "",
    "resumen": [f"RR {r:.1f}; severidad ajustada: {s}; horario con exceso: {h}; palanca: {p}"
                for r, s, h, p in zip(fz["RR_zona"], fz["severidad_ajustada"], fz["horario_con_exceso"], fz["palanca_dominante"])],
    "verificacion_prioritaria": "no aplica",
    "motivo_verificacion": "La verificación por exposición aplica a los puntos críticos.",
})
tp = cel.loc[top_eb.index].reset_index()


def motivo_verificacion(pct, rank_aadt, rank_eb, aadt):
    # La prioridad de un punto depende del volumen de choques, no de la exposición medida, cuando no
    # hay aforo suficiente o cuando el AADT lo manda muy abajo en el ranking (sección 6.3).
    if pct < 0.5:
        return (f"Sin aforo suficiente en la celda ({pct:.0%} de sus siniestros): la prioridad se apoya "
                f"en el volumen de choques.")
    if rank_aadt > 50:
        return (f"El aforo medido ({aadt:,.0f} veh/día) la deja en el puesto {rank_aadt} del ranking: "
                f"su prioridad se apoya en el volumen de choques.")
    return "Se mantiene al priorizar con el aforo medido."


tp["verificacion_prioritaria"] = np.where((tp["pct_aforado"] < 0.5) | (tp["rank_aadt"] > 50), "sí", "no")
tp["motivo_verificacion"] = [motivo_verificacion(p, r, e, a) for p, r, e, a in
                             zip(tp["pct_aforado"], tp["rank_aadt"], tp["rank_EB"], tp["aadt"])]
plantilla_p = pd.DataFrame({
    "tipo": "Punto crítico", "prioridad": tp["rank_EB"], "lugar": tp["lugar"] + " (" + tp["zona"] + ")",
    "graves_por_anio": tp["graves_anio_EB"],
    "ubicacion": [f"{a:.5f}, {b:.5f}" for a, b in zip(tp["lat"], tp["lon"])],
    "resumen": [f"{n} siniestros, {k} graves obs., {f} fatales; {pn:.0f}% nocturno; {pv:.0f}% peatón/ciclista"
                for n, k, f, pn, pv in zip(tp["siniestros"], tp["K"], tp["fatales"], tp["pct_nocturno"], tp["pct_vulnerables"])],
    "verificacion_prioritaria": tp["verificacion_prioritaria"],
    "motivo_verificacion": tp["motivo_verificacion"],
})
plantilla = pd.concat([plantilla_z, plantilla_p], ignore_index=True)
for c in ["confirmado_en_terreno (sí/no)", "intervencion_existente (año y tipo)", "comentarios_observatorio"]:
    plantilla[c] = ""
plantilla["graves_por_anio"] = plantilla["graves_por_anio"].round(1)
plantilla.to_csv("validacion_observatorio.csv", index=False, encoding="utf-8-sig")
plantilla.head(12)

## 8. Salidas para el tablero (Fase 5) y la línea base (Fase 6)

Se exportan tablas planas, listas para conectar a la herramienta de tablero que se elija en la Fase 5. Para la **actualización mensual**, el Observatorio solo necesita reemplazar el archivo de origen y volver a correr `enriquecer_coordenadas.py` y este notebook.

| Archivo | Contenido | Uso |
|---|---|---|
| `riesgo_zonas.csv` | Ficha completa de las 77 zonas (sección 5) | Vista "zonas" del tablero, ranking y mapa |
| `riesgo_zona_horario.csv` | 616 celdas zona × horario con graves observados, estimación EB y RR | Vista "cuándo patrullar" con filtros por zona y horario |
| `puntos_criticos_eb.csv` | Celdas de ~150 m con al menos un grave, con estimación EB, ranking oficial y posición con los proxies alternativos (`rank_conteo`, `rank_epdo`, `rank_aadt`) | Vista "puntos críticos", listado de intervenciones y control de estabilidad del ranking |
| `linea_base_zonas.csv` | Siniestros, con heridos y graves por zona y año 2018–2024 | Línea base para medir el impacto a 12 meses |
| `efectos_modelo_B.csv` | Odds ratios ajustados | Anexo técnico del informe |
| `validacion_observatorio.csv` | Las 10 zonas y los 20 puntos prioritarios con las columnas para completar en terreno, más `verificacion_prioritaria` y su motivo | Planilla de la sección 7.2 |
| `mapa_riesgo_zonas.html` | Mapa interactivo | Presentación al Concejo |

In [ ]:
ficha.reset_index().rename(columns={"index": "zona"}).round(4).to_csv("riesgo_zonas.csv", index=False, encoding="utf-8-sig")
panel[["zona", "horario", "horas", "siniestros", "con_heridos", "graves", "esperado_A", "EB", "graves_anio_EB", "RR_celda"]] \
    .round(4).to_csv("riesgo_zona_horario.csv", index=False, encoding="utf-8-sig")
cel[cel["K"] > 0].sort_values("rank_EB").reset_index()[
    ["rank_EB", "rank_fase3", "rank_conteo", "rank_epdo", "rank_aadt", "estable_top20", "celda", "lugar", "zona",
     "lat", "lon", "siniestros", "K", "fatales", "epdo", "mu", "EB", "graves_anio_EB", "exceso_EB", "pct_nocturno",
     "pct_vulnerables"]].round(4) \
    .to_csv("puntos_criticos_eb.csv", index=False, encoding="utf-8-sig")
linea_base = (dz.groupby(["zona", "anio"]).agg(siniestros=("grave", "size"), con_heridos=("con_heridos", "sum"),
                                               graves=("grave", "sum"), fatales=("fatal", "sum")).reset_index())
linea_base.to_csv("linea_base_zonas.csv", index=False, encoding="utf-8-sig")
efB[["variable", "categoria", "razon", "ic_inf", "ic_sup", "p_valor"]].rename(columns={"razon": "odds_ratio"}) \
    .round(4).to_csv("efectos_modelo_B.csv", index=False, encoding="utf-8-sig")

for a in ["riesgo_zonas.csv", "riesgo_zona_horario.csv", "puntos_criticos_eb.csv", "linea_base_zonas.csv",
          "efectos_modelo_B.csv", "validacion_observatorio.csv", "mapa_riesgo_zonas.html"]:
    print(f"{a:32s} {Path(a).stat().st_size / 1024:8,.0f} KB")

## 9. Conclusiones de la Fase 4

### Viabilidad

**El modelo de riesgo relativo por zona y horario es viable** y se construyó. La zona es el área comunitaria (77 zonas oficiales) y el horario combina franja y tipo de día (8 horarios). Para los puntos concretos (cruces de ~150 m) se usa una corrección Empírico Bayesiana del ranking, porque ahí el volumen de graves no alcanza para estimar tasas confiables.

### Qué se entrega

| Pieza | Qué responde | Desempeño fuera de muestra (2023–2024) |
|---|---|---|
| **Modelo A** (Poisson, zona × horario) | Dónde y cuándo se concentran los graves | Error 58 % menor que el conteo histórico; top 10 % de celdas → 32,5 % de los graves (máx. posible 38,5 %) |
| **Modelo B** (logístico, por siniestro) | Qué vuelve grave a un siniestro y qué zonas son más severas de lo esperable | AUC 0,76, bien calibrado |
| **Ranking EB de puntos críticos** | Qué cruces intervenir, sin sesgo de regresión a la media | Top 50 → +39 % de graves futuros capturados que ordenar por graves observados |
| **Ficha por zona + mapa interactivo** | Resumen para el Concejo | — |
| **Tablas exportadas** | Insumo del tablero (Fase 5) y línea base (Fase 6) | — |

### Hallazgos que agrega esta fase a la Fase 3

1. **Dos zonas se separan del resto:** *Near West Side* y *Austin* tienen ~4 veces los graves de una zona promedio. Junto con *Near North Side* y *Loop* suman ~150 graves por año (el 18 % del total) en 4 de 77 zonas.
2. **Frecuencia y severidad apuntan a horarios distintos.** Por hora, la mayor cantidad de graves se da en las **tardes** (hábiles y de fin de semana). Por siniestro, la mayor letalidad está en la **madrugada y el fin de semana**, y se mantiene al ajustar por alcohol y tipo de choque. Por eso el patrullaje preventivo conviene en las tardes y los controles de alcohol/velocidad, en las noches y madrugadas de fin de semana.
3. **Las causas de fiscalización (alcohol, velocidad, semáforo) siguen siendo las más severas al ajustar**, y el efecto de la velocidad crece.
4. **Ni la iluminación ni la lluvia explican la severidad** una vez que se tiene en cuenta la franja horaria. La hipótesis "de noche con lluvia" se descarta como factor de severidad a escala ciudad.
5. **Hay un "plus de severidad" en el West Side y el South Side.** Zonas como *North Lawndale*, *Garfield Park*, *West Englewood* y *Auburn Gresham* tienen entre 20 % y 30 % más graves de lo que explica su composición, mientras que el centro tiene menos. Es el indicio más fuerte de un problema de **velocidad o diseño vial** y orienta dónde hacer auditorías de infraestructura.
6. **El ranking de puntos críticos de la Fase 3 es robusto** (12 de 15 se mantienen), pero hay que leerlo con la estimación EB: algunos puntos tuvieron "mala suerte" y otros, con mucho volumen, están subestimados.

### Limitaciones del modelo

| # | Limitación | Consecuencia |
|---|---|---|
| 1 | **Sin datos de exposición** (flujo vehicular) | El RR mide dónde se concentra el daño, no la peligrosidad por viaje. Una zona con mucho tránsito puede tener RR alto sin ser "más peligrosa" para cada conductor |
| 2 | **Causa asignada por el oficial**, 30 % sin determinar | Los OR por causa describen la *causa atribuida*. "Sin determinar" se mantiene como categoría para no sesgar la muestra |
| 3 | **Validación temporal de solo 2 años** y posible subregistro de graves en 2024 | La captura de graves futuros puede estar levemente subestimada. Conviene repetir la validación con 2025 completo |
| 4 | **Asociación, no causalidad** | Un OR alto indica dónde mirar, no prueba que intervenir ese factor reduzca los graves en esa magnitud |
| 5 | **El EB usa el total de siniestros como proxy de exposición** | Un cruce con muchos choques leves por congestión puede quedar sobrerrepresentado. La sección 6.3 lo mide con el AADT del portal: el ranking cambia, pero el AADT no anticipa mejor los graves futuros y solo cubre el 59 % de los siniestros. Los puntos que dependen del volumen quedan marcados para verificar en terreno (sección 7) |
| 6 | **Límites administrativos** | Un corredor que cruza varias zonas se reparte entre ellas. Para obra lineal (avenidas) conviene complementar con el ranking de puntos |

### Próximos pasos

- **Validación con el Observatorio Vial:** enviar `validacion_observatorio.csv` y el mapa interactivo, y relevar las preguntas del protocolo (sección 7.1).
- **Fase 5 (tablero):** las tablas exportadas ya tienen la estructura de las vistas propuestas (zonas, zona × horario, puntos críticos, línea base). El tablero puede mostrar el RR y el ranking EB sin recalcular los modelos, y los modelos se reajustan corriendo este notebook al actualizar la base.
- **Fase 6 (intervenciones):** priorizar al menos tres intervenciones cruzando (a) las zonas de mayor RR con severidad ajustada alta, (b) el top EB de puntos críticos y (c) los horarios con exceso propio para los operativos. La línea base por zona y año ya quedó en `linea_base_zonas.csv`.

## Anexo: conceptos estadísticos utilizados

### Riesgo relativo (RR)

**Qué es:** el cociente entre dos tasas. RR = 2 significa "el doble que la referencia"; RR = 0,5, "la mitad". En este notebook la referencia es **el promedio de la ciudad**, así que se lee directamente como "cuántas veces el promedio".

### Regresión de Poisson y *offset*

**Qué es:** el modelo estándar para **conteos** (cantidad de siniestros graves). Supone que el logaritmo de la cantidad esperada es una suma de efectos, así que los efectos se **multiplican**: riesgo de la celda = riesgo de la zona × riesgo del horario. El *offset* (logaritmo de las horas) convierte el conteo en una **tasa por hora**, para que el fin de semana (2 días) se compare de forma justa con la semana hábil (5 días).

### Sobredispersión

**Qué es:** cuando los conteos varían más de lo que el modelo de Poisson supone. Si se ignora, los intervalos de confianza salen demasiado angostos y se "descubren" diferencias que no son reales. Acá se corrige inflando los errores estándar (cuasi-Poisson) y estimando el parámetro $\alpha$ para el método Empírico Bayesiano.

### Regresión logística y *odds ratio* (OR)

**Qué es:** el modelo estándar para una respuesta **sí/no** (grave / no grave). Cada coeficiente se traduce en un **odds ratio ajustado**: cuántas veces aumenta la chance de que el siniestro sea grave por un factor, **con las demás variables fijas**. Cuando el evento es poco frecuente (como acá, 3,4 %), el OR es prácticamente igual al riesgo relativo.

**Crudo vs. ajustado:** el OR crudo mira un factor solo. El ajustado descuenta el efecto de los demás. Si el crudo es alto pero el ajustado ≈ 1, el factor no era la causa, sino que "viajaba" junto con otro (*confusión*).

### AUC y calibración

- **AUC:** si se toma al azar un siniestro grave y uno que no lo fue, es la probabilidad de que el modelo le asigne más riesgo al grave. 0,5 es azar; 0,7–0,8 es aceptable; más de 0,8, bueno.
- **Calibración:** que las probabilidades signifiquen lo que dicen. Si el modelo dice 10 %, alrededor del 10 % de esos siniestros debe resultar grave. Un modelo puede ordenar bien (buen AUC) y estar mal calibrado, o al revés.

### Razón observado/esperado (O/E)

**Qué es:** se compara cuántos graves tuvo una zona con cuántos "debería" haber tenido según su composición (tipos de choque, horarios, causas). O/E = 1,3 significa 30 % más graves de lo esperable. Es la misma lógica con la que se comparan tasas de mortalidad entre hospitales que atienden pacientes distintos.

### Regresión a la media y método Empírico Bayesiano (EB)

**El problema:** si se elige un lugar *porque* tuvo muchos siniestros, lo más probable es que en el período siguiente tenga menos, aunque nadie haga nada, simplemente porque parte de ese exceso fue azar. Esto infla los rankings con lugares "con mala suerte" y hace que las intervenciones parezcan más efectivas de lo que son.

**La solución:** el método EB, recomendado por el *Highway Safety Manual*, estima el riesgo de cada lugar como un **promedio ponderado** entre lo observado y lo esperable para un lugar de sus características. Cuanto menos información tiene el lugar (pocos siniestros), más se apoya en lo esperable; cuanto más información, más confía en lo observado. Es el mismo método que habrá que usar en 12 meses para **evaluar si las intervenciones funcionaron** (comparación antes/después corregida).